In [2]:
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.metrics import accuracy_score, classification_report
import numpy as np



In [3]:
df = pd.read_csv('/content/data.csv')

In [4]:
df.sample(5)

,age,weight,height,income_lpa,smoker,city,occupation,insurance_premium_category
11,33,65,1.67,5.5,False,Jaipur,freelancer,Medium
8,67,115,1.74,0.6,True,Mumbai,retired,High
2,35,68,1.70,6.8,False,Mumbai,freelancer,Medium
14,48,82,1.69,18.0,True,Mumbai,business_owner,High
4,45,76,1.68,12.5,True,Delhi,private_job,Medium


In [5]:
df['occupation'].unique()

array(['student', 'private_job', 'freelancer', 'government_job',
       'unemployed', 'retired', 'business_owner'], dtype=object)

In [6]:
df_feat = df.copy()

In [7]:
# Feature 1 : BMI
df_feat['bmi'] = df_feat['weight'] / (df_feat['height']**2)

In [8]:
# Feature 2 : Age Group
def age_group(age):
  if age < 25:
    return "young"

  elif age < 45:
    return 'adult'

  elif age < 60:
    return "middle_aged"

  return 'senior'

In [9]:
df_feat['age_group'] = df_feat['age'].apply(age_group)

In [10]:
# Feature 3: Lifestyle Risk
def lifestyle_risk(row):
    if row["smoker"] and row["bmi"] > 30:
        return "high"
    elif row["smoker"] or row["bmi"] > 27:
        return "medium"
    else:
        return "low"

In [11]:
df_feat["lifestyle_risk"] = df_feat.apply(lifestyle_risk, axis=1)

In [12]:
tier_1_cities = ["Mumbai", "Delhi", "Bangalore", "Chennai", "Kolkata", "Hyderabad", "Pune"]
tier_2_cities = [
"Jaipur", "Chandigarh", "Indore", "Lucknow", "Patna", "Ranchi", "Visakhapatnam", "Coimbatore", "Bhopal", "Nagpur", "Vadodara", "Surat", "Rajkot", "Jodhpur", "Raipur", "Amritsar", "Varanasi", "Agra", "Dehradun", "Mysore", "Jabalpur", "Guwahati", "Thiruvananthapuram", "Ludhiana", "Nashik", "Allahabad", "Udaipur", "Aurangabad", "Hubli", "Belgaum", "Salem", "Vijayawada", "Tiruchirappalli", "Bhavnagar", "Gwalior", "Dhanbad", "Bareilly", "Aligarh", "Gaya", "Kozhikode", "Warangal", "Kolhapur",
                  "Bilaspur", "Jalandhar", "Noida",
                  "Guntur", "Asansol", "Siliguri"
]


In [13]:
# Feature 4: City Tier

def city_tier(city):
  if city in tier_1_cities:
    return 1
  elif city in tier_2_cities:
    return 2

  else:
    return 3

In [14]:
df_feat["city_tier"] = df_feat['city'].apply(city_tier)

In [15]:
df_feat.drop(columns=['age', 'weight', 'height', 'smoker', 'city'])[['income_lpa', 'occupation', 'bmi', 'age_group', 'lifestyle_risk', 'city_tier', 'insurance_premium_category']].sample(5)



,income_lpa,occupation,bmi,age_group,lifestyle_risk,city_tier,insurance_premium_category
14,18.0,business_owner,28.710479,middle_aged,medium,1,High
10,3.1,student,23.233456,adult,low,1,Low
5,11.9,unemployed,35.430839,middle_aged,high,1,High
15,10.4,private_job,30.071168,middle_aged,medium,1,Medium
18,2.8,retired,27.885187,senior,medium,2,Medium


In [16]:
# select feature and target
x = df_feat[['bmi','age_group','lifestyle_risk','city_tier','income_lpa','occupation']]
y = df_feat['insurance_premium_category']

In [17]:
x

,bmi,age_group,lifestyle_risk,city_tier,income_lpa,occupation
0,21.484375,young,low,1,2.5,student
1,22.773186,adult,low,1,4.2,private_job
2,23.529412,adult,low,1,6.8,freelancer
3,27.688778,adult,medium,1,8.0,government_job
4,26.927438,middle_aged,medium,1,12.5,private_job
5,35.430839,middle_aged,high,1,11.9,unemployed
6,28.731747,middle_aged,medium,1,7.2,private_job
7,22.582709,senior,low,1,3.9,retired
8,37.983882,senior,high,1,0.6,retired
9,26.128611,senior,medium,2,2.0,retired


In [18]:
y

,insurance_premium_category
0,Low
1,Low
2,Medium
3,Low
4,Medium
5,High
6,Medium
7,Medium
8,High
9,High


In [19]:
# define catagorical and numeeric features
catagorical_features = ['age_group','lifestyle_risk','city_tier']
numeric_feature = ['bmi', 'income_lpa']

In [20]:
# create column transform for one hot encoding
preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder() , catagorical_features),
        ('num', 'passthrough', numeric_feature)
    ]
)

In [21]:
# create pipeline with preprocessing and random forest classifier
pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(random_state=42))
])

In [22]:
# split data  and train model
X_train, X_test, y_train, y_test = train_test_split(x, y, test_size=0.2,random_state=1)
pipeline.fit(X_train, y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('cat', OneHotEncoder(),
                                                  ['age_group',
                                                   'lifestyle_risk',
                                                   'city_tier']),
                                                 ('num', 'passthrough',
                                                  ['bmi', 'income_lpa'])])),
                ('classifier', RandomForestClassifier(random_state=42))])

In [23]:
# predict and evaluate
y_pred = pipeline.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy: {accuracy}")

Accuracy: 0.75


In [25]:
import pickle

pickle_model_path = "model.pkl"
with open(pickle_model_path, "wb") as f:
  pickle.dump(pipeline,f)